In [ ]:
import sys, warnings, tempfile
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except AttributeError:
    pass

import time
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image as IPyImage

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
TMP  = Path(tempfile.gettempdir())

from gaffer import config
from gaffer.detection.detector import FootballDetector
from gaffer.detection.team_assigner import TeamAssigner
from gaffer.tracking.tracker import PlayerTracker
from gaffer.tracking.position_store import PositionStore

def show_fig(fname):
    plt.savefig(str(fname), dpi=100, bbox_inches='tight')
    plt.close()
    display(IPyImage(str(fname)))

print('Day 3 — Player Tracking')

In [ ]:
CLIP = ROOT / 'data' / 'test_clips' / 'tactical_playlist_1.mp4'
assert CLIP.exists(), f'Missing: {CLIP}'

cap   = cv2.VideoCapture(str(CLIP))
FPS   = cap.get(cv2.CAP_PROP_FPS) or 25.0
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# 60-second window starting at 30s (skips pre-match intro)
ACTION_START = int(30 * FPS)
N_PROCESS    = int(60 * FPS)   # 1500 frames @ 25fps
N_PROCESS    = min(N_PROCESS, TOTAL - ACTION_START)

print(f'Clip      : {CLIP.name}')
print(f'Resolution: {W}x{H} @ {FPS:.1f} fps  |  {TOTAL} frames total')
print(f'Window    : frames {ACTION_START}–{ACTION_START+N_PROCESS}'
      f'  ({ACTION_START/FPS:.0f}s – {(ACTION_START+N_PROCESS)/FPS:.0f}s)')

In [ ]:
print('Initialising models ...')
t0       = time.time()
detector = FootballDetector(verbose=False)
assigner = TeamAssigner()
tracker  = PlayerTracker(fps=FPS)
store    = PositionStore(fps=FPS)
print(f'  detector  : {detector.model_type}  ({time.time()-t0:.1f}s)')
print(f'  tracker   : thresh={tracker._byte.track_activation_threshold}'
      f'  match={tracker._byte.minimum_matching_threshold}'
      f'  buffer={tracker._byte.max_time_lost}f')
print(f'  detect_every_n = {config.DETECT_EVERY_N_FRAMES}')

# Fit TeamAssigner on 10 frames sampled across the 60s window
cap            = cv2.VideoCapture(str(CLIP))
sample_indices = np.linspace(ACTION_START, ACTION_START + N_PROCESS - 1, 10, dtype=int)
fit_frames, fit_dets = [], []
for idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret: continue
    fit_frames.append(frame)
    fit_dets.append(detector.detect_raw(frame))
cap.release()

assigner.fit(fit_frames, fit_dets)
print(f'  assigner  : fitted on {len(fit_frames)} frames, {assigner.n_fit_samples} jersey crops')
print(f'  cluster→team: {assigner.cluster_to_team}')

In [ ]:
# ── Main pipeline ─────────────────────────────────────────────────────────────
# Per-frame data for analysis
per_frame_active   = []          # int: confirmed track count
per_frame_t0       = []          # int: T0 player count
per_frame_t1       = []          # int: T1 player count
per_frame_dets     = []          # List[Detection]: stored for failure analysis

cap     = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, ACTION_START)
timings = []

for local_idx in range(N_PROCESS):
    ret, frame = cap.read()
    if not ret: break
    gidx = ACTION_START + local_idx
    t0   = time.time()

    dets = detector.detect(frame, gidx)
    dets = assigner.assign(frame, dets)

    if gidx == detector._last_detect_idx:      # detection ran this frame
        dets = tracker.update(dets)
    else:                                       # skip frame — carry forward
        dets = tracker.carry_forward()

    store.update(gidx, dets)
    timings.append(time.time() - t0)

    per_frame_active.append(sum(1 for d in dets if d.track_id >= 0))
    per_frame_t0.append(sum(1 for d in dets if d.team_id == 0))
    per_frame_t1.append(sum(1 for d in dets if d.team_id == 1))
    per_frame_dets.append(dets)

cap.release()
print(f'Processed {N_PROCESS} frames in {sum(timings):.1f}s  ({np.mean(timings)*1000:.1f} ms/frame mean)')

In [ ]:
# ── Track stability metrics ───────────────────────────────────────────────────
track_ids       = store.track_ids
n_unique        = len(track_ids)
lengths         = [store.track_length(t) for t in track_ids]
mean_len        = np.mean(lengths) if lengths else 0
max_len         = max(lengths) if lengths else 0

# ID persistence: fraction of span actually observed (1.0 = never dropped)
persistences    = [p for t in track_ids if (p := store.persistence(t)) is not None]
mean_pers       = np.mean(persistences) if persistences else 0

# «Long-lived» tracks: visible for >= 10 seconds (250 frames at 25fps)
TEN_SEC_FRAMES  = int(10 * FPS)
long_tracks     = [t for t in track_ids if store.track_length(t) >= TEN_SEC_FRAMES]

# ID switches proxy: tracks that went missing then reappeared (gap > 1s)
GAP_THRESH      = int(FPS)   # 1-second gap = dropped track came back
id_switches     = 0
fragmented_ids  = []
for tid in track_ids:
    hist = store.get_track_history(tid)
    frames = [e.frame_idx for e in hist]
    gaps   = [frames[i+1] - frames[i] for i in range(len(frames)-1)]
    if any(g > GAP_THRESH for g in gaps):
        id_switches += 1
        fragmented_ids.append(tid)

mean_active = np.mean(per_frame_active)
std_active  = np.std(per_frame_active)

print('=== Track Stability ===')
print(f'  Unique track IDs       : {n_unique}')
print(f'  Mean track length      : {mean_len:.1f} frames ({mean_len/FPS:.1f}s)')
print(f'  Longest track          : {max_len} frames ({max_len/FPS:.1f}s)')
print(f'  Tracks >= 10s          : {len(long_tracks)}')
print(f'  Mean persistence       : {mean_pers:.2%}')
print(f'  ID switches (gap>1s)   : {id_switches}')
print(f'  Mean active/frame      : {mean_active:.1f} +/- {std_active:.1f}')

# Success criterion: a player visible 10+ seconds usually keeps the same ID.
# We verify: long_tracks exist AND their persistence is high.
long_persis = [store.persistence(t) for t in long_tracks if store.persistence(t) is not None]
if long_persis:
    print(f'\n  Long-track persistence : {np.mean(long_persis):.2%} (mean over {len(long_persis)} tracks)')

verdict = 'PASS' if (len(long_tracks) >= 3 and mean_pers >= 0.60) else 'WARN'
print(f'\nVERDICT: {verdict}')

In [ ]:
# ── Failure-case analysis ─────────────────────────────────────────────────────
# 1. Camera pans: frames where active track count drops sharply
window   = int(FPS)   # 1-second rolling mean
rolling  = np.convolve(per_frame_active, np.ones(window)/window, mode='same')
drop     = np.array(per_frame_active, float) - rolling
pan_frames = np.where(drop < -3)[0].tolist()   # active tracks fell >3 below 1s mean

# 2. Player overlaps: frames with at least one pair of high-IoU bboxes
def bbox_iou(a, b):
    ax1,ay1,ax2,ay2 = a;  bx1,by1,bx2,by2 = b
    ix1 = max(ax1,bx1); iy1 = max(ay1,by1)
    ix2 = min(ax2,bx2); iy2 = min(ay2,by2)
    if ix2 <= ix1 or iy2 <= iy1: return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    return inter / (ua + ub - inter)

overlap_frames = []
for fi, dets in enumerate(per_frame_dets):
    players = [d for d in dets if d.class_name in ('player','goalkeeper','person')]
    found = False
    for i in range(len(players)):
        for j in range(i+1, len(players)):
            if bbox_iou(players[i].bbox, players[j].bbox) > 0.20:
                found = True; break
        if found: break
    if found: overlap_frames.append(fi)

# 3. Temporary disappearances: all fragmented tracks
disapp_count = len(fragmented_ids)

print('=== Failure Cases ===')
print(f'  Camera-pan candidate frames  : {len(pan_frames)}'
      f'  ({100*len(pan_frames)/N_PROCESS:.1f}%)')
print(f'  Overlap frames (IoU>0.20)    : {len(overlap_frames)}'
      f'  ({100*len(overlap_frames)/N_PROCESS:.1f}%)')
print(f'  Tracks with disappearances   : {disapp_count}'
      f'  (gap > {GAP_THRESH} frames)')
if fragmented_ids:
    print(f'  Fragmented track IDs         : {fragmented_ids}')

In [ ]:
# ── Plot 1: active tracks + team counts ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
xs = np.arange(N_PROCESS) / FPS   # x-axis in seconds

axes[0].plot(xs, per_frame_active, lw=0.8, color='steelblue', label='active tracks')
axes[0].axhline(mean_active, color='orange', ls='--', lw=1, label=f'mean={mean_active:.1f}')
for pf in pan_frames[:30]:           # mark first 30 pan candidates
    axes[0].axvline(pf/FPS, color='red', alpha=0.15, lw=0.5)
axes[0].set_title('Active confirmed tracks per frame  (red ticks = pan candidates)')
axes[0].set_ylabel('# tracks'); axes[0].set_ylim(0, 20)
axes[0].legend(loc='upper right'); axes[0].set_xlabel('Time (s)')

axes[1].plot(xs, per_frame_t0, lw=0.8, color='red',  label='T0 (red)')
axes[1].plot(xs, per_frame_t1, lw=0.8, color='blue', label='T1 (blue)')
axes[1].set_title('Per-frame team player count')
axes[1].set_ylabel('# players'); axes[1].set_ylim(0, 14)
axes[1].legend(loc='upper right'); axes[1].set_xlabel('Time (s)')

plt.tight_layout()
show_fig(TMP / 'd3_counts.png')

In [ ]:
# ── Plot 2: Gantt-style track timeline ───────────────────────────────────────
# Assign majority team colour to each track
team_votes = defaultdict(lambda: defaultdict(int))
for dets in per_frame_dets:
    for d in dets:
        if d.track_id >= 0 and d.team_id >= 0:
            team_votes[d.track_id][d.team_id] += 1

track_team = {}
for tid, votes in team_votes.items():
    track_team[tid] = max(votes, key=votes.get)

COLORS = {0: 'red', 1: 'royalblue', -1: 'grey'}

sorted_tracks = sorted(track_ids, key=lambda t: store.first_frame(t) or 0)
fig, ax = plt.subplots(figsize=(14, max(3, len(sorted_tracks) * 0.45)))

for row, tid in enumerate(sorted_tracks):
    hist   = store.get_track_history(tid)
    frames = [(e.frame_idx - ACTION_START) / FPS for e in hist]
    team   = track_team.get(tid, -1)
    c      = COLORS.get(team, 'grey')
    ax.scatter(frames, [row]*len(frames), s=4, c=c, alpha=0.8)
    pers = store.persistence(tid)
    label = f'#{tid}  ({store.track_length(tid)}f  {pers:.0%})' if pers else f'#{tid}'
    ax.text(-0.5, row, label, fontsize=7, va='center', ha='right')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Track (sorted by start)')
ax.set_title(f'Track timelines  |  {n_unique} unique IDs  |  red=T0, blue=T1')
ax.set_xlim(-3, N_PROCESS/FPS)
ax.set_ylim(-1, len(sorted_tracks))

legend = [mpatches.Patch(color='red', label='T0'),
          mpatches.Patch(color='royalblue', label='T1'),
          mpatches.Patch(color='grey', label='unknown')]
ax.legend(handles=legend, loc='upper right')
plt.tight_layout()
show_fig(TMP / 'd3_timelines.png')

In [ ]:
# ── Produce annotated MP4 ─────────────────────────────────────────────────────
OUTPUT = ROOT / 'outputs' / 'day3_tracking_demo.mp4'
OUTPUT.parent.mkdir(exist_ok=True)

# 30-colour palette — enough for any realistic number of simultaneous tracks
PALETTE = [
    (255,50,50),(50,50,255),(50,200,50),(200,50,200),(50,200,200),
    (200,200,50),(150,0,150),(0,150,150),(150,150,0),(80,0,200),
    (0,80,200),(200,80,0),(220,120,0),(0,120,220),(120,0,220),
    (180,90,90),(90,180,90),(90,90,180),(180,180,90),(90,180,180),
    (180,90,180),(255,140,0),(0,255,140),(140,0,255),(255,0,140),
    (0,140,255),(140,255,0),(255,80,80),(80,255,80),(80,80,255),
]

TEAM_COLORS = {0: (0,0,220), 1: (220,0,0)}   # BGR: red=T0, blue=T1
BALL_COLOR  = (0,220,220)
UNKNOWN_CLR = (180,180,180)

def tid_color(track_id):
    return PALETTE[track_id % len(PALETTE)] if track_id >= 0 else UNKNOWN_CLR

def draw_frame(frame, dets, local_idx):
    out = frame.copy()
    for d in dets:
        x1,y1,x2,y2 = d.bbox
        # ── Ball ──
        if d.class_name == 'ball':
            cv2.rectangle(out,(x1,y1),(x2,y2),BALL_COLOR,2)
            cv2.putText(out,f'ball {d.confidence:.2f}',(x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.42,BALL_COLOR,1)
            continue
        # ── Player/GK ── thick box in TEAM colour, thin label in TRACK colour
        box_c = TEAM_COLORS.get(d.team_id, UNKNOWN_CLR)
        cv2.rectangle(out,(x1,y1),(x2,y2), box_c, 2)
        lbl   = f'#{d.track_id}' if d.track_id >= 0 else '?'
        tlbl  = f'T{d.team_id}'  if d.team_id  >= 0 else ''
        txt_c = tid_color(d.track_id)
        cv2.putText(out,f'{lbl} {tlbl}',(x1,max(y1-5,10)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.45,txt_c,1,cv2.LINE_AA)
    # HUD
    gidx   = ACTION_START + local_idx
    ts     = gidx / FPS
    active = sum(1 for d in dets if d.track_id >= 0)
    hud_lines = [
        f'Frame {gidx}  t={ts:.1f}s',
        f'Active tracks: {active}',
    ]
    for i, l in enumerate(hud_lines):
        cv2.putText(out, l, (10, 22+i*22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)
    return out

print('Writing annotated video ...')
fourcc      = cv2.VideoWriter_fourcc(*'mp4v')
writer      = cv2.VideoWriter(str(OUTPUT), fourcc, FPS, (W, H))
cap         = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, ACTION_START)

# Fresh detector/tracker/assigner for the video pass
det2  = FootballDetector(verbose=False)
ass2  = TeamAssigner()
ass2.fit(fit_frames, fit_dets)   # reuse the same fit frames
trk2  = PlayerTracker(fps=FPS)

SNAP_FRAMES = set(np.linspace(0, N_PROCESS-1, 9, dtype=int))
snaps       = {}

for li in range(N_PROCESS):
    ret, frame = cap.read()
    if not ret: break
    gi   = ACTION_START + li
    dets = det2.detect(frame, gi)
    dets = ass2.assign(frame, dets)
    if gi == det2._last_detect_idx:
        dets = trk2.update(dets)
    else:
        dets = trk2.carry_forward()
    out = draw_frame(frame, dets, li)
    writer.write(out)
    if li in SNAP_FRAMES:
        snaps[li] = out.copy()

cap.release()
writer.release()
print(f'Saved: {OUTPUT.name}  ({OUTPUT.stat().st_size/1024**2:.1f} MB)')

In [ ]:
# ── Sample frames ─────────────────────────────────────────────────────────────
keys = sorted(snaps.keys())
rows = (len(keys) + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 4))
axes = axes.flatten()
for i, k in enumerate(keys):
    axes[i].imshow(cv2.cvtColor(snaps[k], cv2.COLOR_BGR2RGB))
    axes[i].set_title(f't={( ACTION_START+k)/FPS:.1f}s')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
show_fig(TMP / 'd3_snaps.png')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=== Day 3 Summary ===')
print(f'  Clip          : {CLIP.name}')
print(f'  Window        : {N_PROCESS/FPS:.0f}s  ({N_PROCESS} frames)')
print(f'  Speed         : {np.mean(timings)*1000:.1f} ms/frame')
print()
print('  Tracking:')
print(f'    Unique IDs      : {n_unique}')
print(f'    Mean length     : {mean_len:.1f}f  ({mean_len/FPS:.1f}s)')
print(f'    Longest track   : {max_len}f  ({max_len/FPS:.1f}s)')
print(f'    10s+ tracks     : {len(long_tracks)}')
print(f'    Mean persistence: {mean_pers:.1%}')
print(f'    ID switches     : {id_switches}')
print(f'    Mean active/f   : {mean_active:.1f}')
print()
print('  Failure cases:')
print(f'    Pan frames      : {len(pan_frames)}')
print(f'    Overlap frames  : {len(overlap_frames)}')
print(f'    Disappearances  : {disapp_count}')
print()
print(f'  VERDICT: {verdict}')
print(f'  Output : outputs/day3_tracking_demo.mp4  ({OUTPUT.stat().st_size/1024**2:.1f} MB)')